In [1]:

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    pairwise_distances_argmin,
    classification_report,
    accuracy_score,
    silhouette_score
)


In [2]:

# Load datasets
train = pd.read_csv("../training_image_features_grouped.csv")
test = pd.read_csv("../testing_image_features_grouped.csv")

TARGET = "Condition_Class"
CLASSES = ["Excellent", "Good", "Fair", "Poor"]

print("Training Dataset Shape:", train.shape)
print("Testing Dataset Shape:", test.shape)

print("\nTraining Dataset:")
display(train.head())

print("\nTesting Dataset:")
display(test.head())


Training Dataset Shape: (11465, 43)
Testing Dataset Shape: (2644, 43)

Training Dataset:


,Sample_ID,Image_ID,Material_Type,Material_Subtype,Source,Exposure_Type,Crack_Level,Surface_Damage,Breakage_Level,Discoloration_Level,...,dark_pixel_ratio,bright_pixel_ratio,mean_hue,mean_saturation,saturation_std,contour_area,contour_perimeter,circularity,solidity,extent
0,TRAINING_00013,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,2,Moderate,...,0.013321,0.141387,16.121604,106.880770,22.460271,16131.0,745.470123,0.364763,0.928938,0.768143
1,TRAINING_00014,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,4.0,3,High,...,0.017225,0.326960,19.273151,94.502765,23.173802,6688.0,441.102593,0.431943,0.901408,0.557891
2,TRAINING_00015,10.jpg,tile,Standard,Demolition_Site,Exterior_Weathered,3,3.5,4,High,...,0.118015,0.170967,29.613107,51.448496,24.297176,10975.5,720.884337,0.265401,0.810628,0.683917
3,TRAINING_00016,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.009063,0.008142,13.644708,119.697837,26.109004,19055.5,769.251868,0.404663,0.893785,0.598101
4,TRAINING_00017,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.000417,0.306625,19.245853,95.316954,25.873127,7753.5,720.364566,0.187760,0.805977,0.488502



Testing Dataset:


,Sample_ID,Image_ID,Material_Type,Material_Subtype,Source,Exposure_Type,Crack_Level,Surface_Damage,Breakage_Level,Discoloration_Level,...,dark_pixel_ratio,bright_pixel_ratio,mean_hue,mean_saturation,saturation_std,contour_area,contour_perimeter,circularity,solidity,extent
0,TRAINING_00144,11.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,1,1.2,2,Low,...,0.240314,0.000000,35.286902,88.061490,21.314621,9865.0,464.308656,0.575034,0.944245,0.788821
1,TRAINING_00145,11.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,2.5,2,Low,...,0.184474,0.000000,60.782451,24.505001,12.455336,5562.0,564.357426,0.219448,0.782004,0.676972
2,TRAINING_00146,11.jpg,tile,Standard,Demolition_Site,Exterior_Weathered,4,3.8,3,High,...,0.268345,0.239161,53.394266,59.908019,22.230752,10227.0,694.666081,0.266321,0.889111,0.525162
3,TRAINING_00147,11.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,1,1.2,2,Low,...,0.374775,0.000000,47.548829,83.334865,36.202832,6660.5,402.374672,0.516958,0.937307,0.611616
4,TRAINING_00148,11.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,4.0,2,High,...,0.001208,0.213500,19.069646,90.216720,19.077536,5071.5,365.462984,0.477155,0.898804,0.686916


In [3]:

# Exclude target-related scores
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

# Select numeric image features
from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

print("Number of Features:", len(features))
print("\nSelected Features:")
print(features)


Number of Features: 26

Selected Features:
['Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Deformation_Level', 'pixel_img_width', 'pixel_img_height', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']


In [4]:

# Extract selected features
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_test = test[features].replace([np.inf, -np.inf], np.nan)

# Handle missing values
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)
print("Preprocessing Completed Successfully")


Training Data Shape: (11465, 26)
Testing Data Shape: (2644, 26)
Preprocessing Completed Successfully


In [5]:

# Initialize K-Means model
model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

# Fit model on training data
train_clusters = model.fit_predict(X_train)

# Predict clusters for testing data
test_clusters = model.predict(X_test)

print("K-Means Clustering Completed")
print("\nCluster Distribution:")
print(pd.Series(train_clusters).value_counts().sort_index())


K-Means Clustering Completed

Cluster Distribution:
0    3015
1    2931
2    1404
3    4115
Name: count, dtype: int64


In [6]:

# Map clusters to majority condition classes
cluster_mapping = {}

for cluster in range(4):
    labels = train.loc[
        train_clusters == cluster, TARGET
    ]

    if len(labels) > 0:
        cluster_mapping[cluster] = labels.mode().iloc[0]
    else:
        cluster_mapping[cluster] = train[TARGET].mode().iloc[0]

print("Cluster to Condition Mapping:")
print(cluster_mapping)


Cluster to Condition Mapping:
{0: 'Fair', 1: 'Fair', 2: 'Fair', 3: 'Fair'}


In [7]:

# Predict testing classes
y_pred = np.array([
    cluster_mapping[c] for c in test_clusters
])

y_test = test[TARGET].astype(str)

print("Predicted Classes:")
print(pd.Series(y_pred).value_counts())

print("\nActual Classes:")
print(y_test.value_counts())


Predicted Classes:
Fair    2644
Name: count, dtype: int64

Actual Classes:
Condition_Class
Fair         1788
Good          586
Poor          238
Excellent      32
Name: count, dtype: int64


In [8]:

# Evaluate K-Means
print("K-Means Clustering")

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    zero_division=0
))

# Evaluate cluster separation
silhouette = silhouette_score(
    X_train,
    train_clusters,
    sample_size=2000,
    random_state=42
)

print("Silhouette Score:", silhouette)

print("Inertia:", model.inertia_)


K-Means Clustering
Accuracy: 0.6762481089258698

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        32
        Good       0.00      0.00      0.00       586
        Fair       0.68      1.00      0.81      1788
        Poor       0.00      0.00      0.00       238

    accuracy                           0.68      2644
   macro avg       0.17      0.25      0.20      2644
weighted avg       0.46      0.68      0.55      2644

Silhouette Score: 0.15740131571840688
Inertia: 176854.9003419055


In [9]:
# Clustering is unsupervised: the accuracy above comes from mapping each cluster to its
# majority class, so compare it with the majority-class baseline and use label-free metrics.
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, f1_score)
_y = test[TARGET].astype(str)
_X = X_train_pca if "X_train_pca" in globals() else X_train
print("Majority-class baseline accuracy:", round(_y.value_counts(normalize=True).max(), 4))
print("Macro F1 of cluster-to-class mapping:", round(f1_score(_y, y_pred, average="macro", zero_division=0), 4))
print("Adjusted Rand Index vs true labels (test):", round(adjusted_rand_score(_y, test_clusters), 4))
print("Normalized Mutual Information (test):", round(normalized_mutual_info_score(_y, test_clusters), 4))
print("Silhouette (train sample):", round(silhouette_score(_X, train_clusters, sample_size=2000, random_state=42), 4))


Majority-class baseline accuracy: 0.6762
Macro F1 of cluster-to-class mapping: 0.2017
Adjusted Rand Index vs true labels (test): 0.0721
Normalized Mutual Information (test): 0.1
Silhouette (train sample): 0.1574


In [10]:

# K-MEANS CLUSTER VISUALIZATION

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load training dataset
train = pd.read_csv("../training_image_features_grouped.csv")

# Select numerical features
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

# Clean the dataset
X = train[features].replace([np.inf, -np.inf], np.nan)

# Handle missing values
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Apply K-Means clustering
kmeans_vis = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_vis.fit_predict(X)

# Select samples for visualization
np.random.seed(42)

sample_size = min(3000, len(X))

sample_indices = np.random.choice(
    len(X),
    sample_size,
    replace=False
)

X_sample = X[sample_indices]
sample_labels = cluster_labels[sample_indices]

# Reduce dimensions using PCA
pca_vis = PCA(n_components=2)

X_2d = pca_vis.fit_transform(X_sample)

# Transform cluster centroids
centroids_2d = pca_vis.transform(
    kmeans_vis.cluster_centers_
)

# Plot clusters
plt.figure(figsize=(10, 7))

for cluster in range(4):

    plt.scatter(
        X_2d[sample_labels == cluster, 0],
        X_2d[sample_labels == cluster, 1],
        label=f"Cluster {cluster}",
        alpha=0.5,
        s=25
    )

# Plot centroids
plt.scatter(
    centroids_2d[:, 0],
    centroids_2d[:, 1],
    marker="X",
    s=250,
    edgecolors="black",
    linewidths=2,
    label="Centroids"
)

plt.title("K-Means Clustering Visualization", fontsize=14)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("K-Means visualization completed successfully!")
print("Number of clusters:", 4)
print("Samples visualized:", sample_size)
print("Explained variance:", pca_vis.explained_variance_ratio_.sum())


K-Means visualization completed successfully!
Number of clusters: 4
Samples visualized: 3000
Explained variance: 0.4665248297628871


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_13208\2726078037.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load training dataset
train_graph = pd.read_csv("../training_image_features_grouped.csv")

# Select numerical features and exclude target-related columns
excluded_columns = [
    "Condition_Class",
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
feature_columns_graph = list(ORIGINAL_FEATURES)

X_graph = train_graph[feature_columns_graph].replace(
    [np.inf, -np.inf], np.nan
)

# Handle missing values
imputer_graph = SimpleImputer(strategy="median")
X_graph = imputer_graph.fit_transform(X_graph)

# Scale features
scaler_graph = StandardScaler()
X_graph_scaled = scaler_graph.fit_transform(X_graph)

# Train K-Means
graph_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

graph_train_clusters = graph_kmeans.fit_predict(X_graph_scaled)

# Count samples in each cluster
cluster_counts = np.bincount(graph_train_clusters, minlength=4)

# Plot graph
plt.figure(figsize=(8, 5))

plt.bar(
    [f"Cluster {i}" for i in range(4)],
    cluster_counts
)

plt.title("K-Means: Number of Samples in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Cluster counts:", cluster_counts)


Cluster counts: [3015 2931 1404 4115]


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_13208\939101827.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

clusters = KMeans(
    n_clusters=4, random_state=42, n_init=10
).fit_predict(X)

distribution = pd.crosstab(
    pd.Series(clusters, name="Cluster"),
    train_graph["Condition_Class"].reset_index(drop=True)
)

distribution = distribution.reindex(
    columns=["Excellent", "Good", "Fair", "Poor"],
    fill_value=0
)

distribution.plot(kind="bar", figsize=(10, 6))

plt.title("K-Means: Actual Condition Distribution in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.legend(title="Condition")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_13208\3177437610.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

train_graph = pd.read_csv("../training_image_features_grouped.csv")
test_graph = pd.read_csv("../testing_image_features_grouped.csv")

target = "Condition_Class"
classes = ["Excellent", "Good", "Fair", "Poor"]

train_clusters = model.predict(X_train)
test_clusters = model.predict(X_test)

y_train = train[target].reset_index(drop=True)

excluded = [
    target, "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X_train = train_graph[features].replace([np.inf, -np.inf], np.nan)
X_test = test_graph.reindex(columns=features).replace(
    [np.inf, -np.inf], np.nan
)

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_clusters = KMeans(
    n_clusters=4, random_state=42, n_init=10
).fit_predict(X_train)

test_clusters_model = KMeans(
    n_clusters=4, random_state=42, n_init=10
)
test_clusters_model.fit(X_train)

test_clusters = test_clusters_model.predict(X_test)

mapping = {}
y_train = train_graph[target].reset_index(drop=True)

for cluster in range(4):
    labels = y_train[train_clusters == cluster]
    mapping[cluster] = labels.mode().iloc[0] if len(labels) else "Fair"

predictions = pd.Series(test_clusters).map(mapping)

cm = confusion_matrix(
    test_graph[target], predictions, labels=classes
)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(
    cm, display_labels=classes
).plot(ax=ax, cmap="Blues", values_format="d")

plt.title("K-Means Confusion Matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_13208\1976110090.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
